<a href="https://colab.research.google.com/github/Oruntu-Tanima-Proje/otProje/blob/main/notebooks/07_gradcam_analizi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔥 07 - Grad-CAM Analizi

## Açıklanabilir Yapay Zeka (XAI)

Bu notebook, **Grad-CAM** (Gradient-weighted Class Activation Mapping) tekniğiyle
3 modelin "yaprağın hangi bölgesine bakarak karar verdiğini" görselleştirir.

### Grad-CAM Nedir?
Grad-CAM, derin öğrenme modellerinin **kara kutu** olmaktan çıkmasını sağlayan
bir görselleştirme tekniğidir. Modelin tahminini yaparken görüntünün hangi
bölgelerine odaklandığını **ısı haritası (heatmap)** olarak gösterir.

### Renk Skalası
- 🔴 **Kırmızı/Sarı**: Modelin DİKKAT verdiği bölgeler (önemli)
- 🔵 **Mavi/Yeşil**: Modelin önemsemediği bölgeler

### Bu Analizin Önemi
- Model **doğru sebepten** doğru cevabı veriyor mu?
- Hastalıklı bölgeye mi bakıyor, arka plana mı?
- 3 model **aynı görüntüde** nereye odaklanıyor?

### Akademik Değer
**Açıklanabilir AI (XAI)** modern derin öğrenmenin önemli bir araştırma alanıdır.
Tıp, tarım, otonom sürüş gibi kritik uygulamalarda model kararlarının
doğrulanabilir olması şarttır.

In [ ]:
# ============================================================
# 1. HAZIRLIK
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as efficient_preprocess

IMG_SIZE = 224
drive_proje = "/content/drive/MyDrive/Domates_Projesi"

# Veriyi Drive'dan kopyala
if not os.path.exists("tomato_data"):
    print("📦 Veri seti Drive'dan kopyalanıyor...")
    shutil.copytree(f"{drive_proje}/data", "tomato_data")

# Modelleri Drive'dan kopyala
os.makedirs("models", exist_ok=True)
for model_file in os.listdir(f"{drive_proje}/models"):
    src = f"{drive_proje}/models/{model_file}"
    dst = f"models/{model_file}"
    if not os.path.exists(dst):
        shutil.copy(src, dst)

# 3 modeli yükle
print("📥 Modeller yükleniyor...")
mobilenet_best = load_model('models/mobilenetv2_final.keras')
resnet_best = load_model('models/resnet50_final.keras')
eff_best = load_model('models/efficientnetb0_final.keras')

print("\n✅ Hazırlık tamamlandı")

# Her model için son conv layer adlarını bul
print("\n🔍 Modellerin son conv katmanları:")
last_layers = {}
for model_name, model_obj in [
    ('MobileNetV2', mobilenet_best),
    ('ResNet50', resnet_best),
    ('EfficientNetB0', eff_best)
]:
    last_conv = None
    for layer in reversed(model_obj.layers):
        if 'conv' in layer.name.lower() or 'Conv' in layer.name:
            last_conv = layer.name
            break
    last_layers[model_name] = last_conv
    print(f"   {model_name}: {last_conv}")

In [ ]:
# ============================================================
# 2. GRAD-CAM YARDIMCI FONKSİYONLARI
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image


def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """
    Grad-CAM ısı haritası oluşturur.

    Args:
        img_array: Ön işlenmiş görüntü (1, H, W, 3)
        model: Eğitilmiş model
        last_conv_layer_name: Son convolutional katmanın adı
        pred_index: Tahmin edilen sınıf indeksi (None = otomatik)

    Returns:
        Heatmap (numpy array, 0-1 arası)
    """
    grad_model = tf.keras.models.Model(
        model.inputs, [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()


def overlay_heatmap_on_image(img_path, heatmap, alpha=0.4):
    """Heatmap'i orijinal görüntü üzerine bindiri."""
    img = np.array(Image.open(img_path).resize((IMG_SIZE, IMG_SIZE)))

    heatmap_resized = np.uint8(255 * heatmap)
    jet = plt.colormaps['jet']
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap_resized]

    jet_heatmap = Image.fromarray(np.uint8(jet_heatmap * 255)).resize((IMG_SIZE, IMG_SIZE))
    jet_heatmap = np.array(jet_heatmap)

    superimposed = jet_heatmap * alpha + img * (1 - alpha)
    return np.uint8(superimposed), img


print("✅ Grad-CAM fonksiyonları hazır")
print("   • make_gradcam_heatmap() — Isı haritası oluşturur")
print("   • overlay_heatmap_on_image() — Görüntü üzerine bindiri")

In [ ]:
# ============================================================
# 3. HASTALIKLI YAPRAKLAR - 3 MODEL KARŞILAŞTIRMASI
# ============================================================

import random

random.seed(42)

# Test setinden hastalıklı sınıfları al
test_path = "tomato_data/test"
classes = sorted([c for c in os.listdir(test_path) if c != "Tomato___healthy"])

# Türkçe sınıf isimleri
class_tr = {
    "Tomato___Bacterial_spot": "Bakteriyel Leke",
    "Tomato___Early_blight": "Erken Yaprak Yanıklığı",
    "Tomato___Late_blight": "Geç Yaprak Yanıklığı",
    "Tomato___Leaf_Mold": "Yaprak Küfü",
    "Tomato___Septoria_leaf_spot": "Septorya Yaprak Lekesi",
    "Tomato___Spider_mites Two-spotted_spider_mite": "Kırmızı Örümcek",
    "Tomato___Target_Spot": "Hedef Leke",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus": "Sarı Yaprak Kıvırcıklığı",
    "Tomato___Tomato_mosaic_virus": "Mozaik Virüsü",
    "Tomato___healthy": "Sağlıklı"
}

# Modeller ve preprocessing fonksiyonları
model_configs = [
    ('MobileNetV2', mobilenet_best, lambda x: x / 255.0),
    ('ResNet50', resnet_best, resnet_preprocess),
    ('EfficientNetB0', eff_best, efficient_preprocess)
]

# 3 farklı hastalıklı sınıftan örnek seç
selected_classes = random.sample(classes, 3)

fig, axes = plt.subplots(3, 4, figsize=(18, 14))

for row, cls in enumerate(selected_classes):
    cls_path = f"{test_path}/{cls}"
    img_file = random.choice(os.listdir(cls_path))
    img_path = f"{cls_path}/{img_file}"

    # Orijinal görüntü
    orig_img = np.array(Image.open(img_path).resize((IMG_SIZE, IMG_SIZE)))
    axes[row, 0].imshow(orig_img)
    axes[row, 0].set_title(f"Orijinal\n{class_tr[cls]}", fontsize=11, fontweight='bold')
    axes[row, 0].axis('off')

    # Her model için Grad-CAM
    for col, (model_name, model_obj, preprocess_fn) in enumerate(model_configs, 1):
        img_loaded = np.array(Image.open(img_path).resize((IMG_SIZE, IMG_SIZE)))
        img_array = np.expand_dims(img_loaded.astype(np.float32), axis=0)
        img_processed = preprocess_fn(img_array.copy())

        try:
            heatmap = make_gradcam_heatmap(img_processed, model_obj, last_layers[model_name])
            superimposed, _ = overlay_heatmap_on_image(img_path, heatmap, alpha=0.5)

            pred = model_obj.predict(img_processed, verbose=0)
            pred_class = np.argmax(pred[0])
            confidence = pred[0][pred_class] * 100

            axes[row, col].imshow(superimposed)
            axes[row, col].set_title(f"{model_name}\nTahmin: %{confidence:.1f}", fontsize=11)
            axes[row, col].axis('off')
        except Exception as e:
            axes[row, col].text(0.5, 0.5, f"Hata: {str(e)[:50]}",
                                ha='center', va='center', transform=axes[row, col].transAxes)
            axes[row, col].axis('off')

plt.suptitle('Grad-CAM: 3 Model Aynı Yaprağa Nasıl Bakıyor?',
             fontsize=15, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('gradcam_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Grad-CAM karşılaştırması kaydedildi")
print("\nYorum:")
print("   🔴 Sıcak renkler (kırmızı/sarı) = Modelin DİKKATİ BURADA")
print("   🔵 Soğuk renkler (mavi)         = Modelin önemsemediği bölgeler")
print("   ✅ İdeal: Hastalık belirtisi olan bölgelere odaklanmalı")

In [ ]:
# ============================================================
# 4. SAĞLIKLI YAPRAK ANALİZİ
# ============================================================

healthy_path = f"{test_path}/Tomato___healthy"
healthy_img_file = random.choice(os.listdir(healthy_path))
healthy_img_path = f"{healthy_path}/{healthy_img_file}"

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Orijinal
orig_img = np.array(Image.open(healthy_img_path).resize((IMG_SIZE, IMG_SIZE)))
axes[0].imshow(orig_img)
axes[0].set_title("Orijinal\nSağlıklı Yaprak", fontsize=12, fontweight='bold')
axes[0].axis('off')

# Her model için Grad-CAM
for col, (model_name, model_obj, preprocess_fn) in enumerate(model_configs, 1):
    img_loaded = np.array(Image.open(healthy_img_path).resize((IMG_SIZE, IMG_SIZE)))
    img_array = np.expand_dims(img_loaded.astype(np.float32), axis=0)
    img_processed = preprocess_fn(img_array.copy())

    heatmap = make_gradcam_heatmap(img_processed, model_obj, last_layers[model_name])
    superimposed, _ = overlay_heatmap_on_image(healthy_img_path, heatmap, alpha=0.5)

    pred = model_obj.predict(img_processed, verbose=0)
    pred_class = np.argmax(pred[0])
    confidence = pred[0][pred_class] * 100

    axes[col].imshow(superimposed)
    axes[col].set_title(f"{model_name}\nTahmin: %{confidence:.1f}", fontsize=11)
    axes[col].axis('off')

plt.suptitle('Grad-CAM: Sağlıklı Yaprak Analizi',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('gradcam_healthy.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Sağlıklı yaprak Grad-CAM kaydedildi")
print("\nYorum: Sağlıklı yaprakta modeller arasında ilginç fark görüyoruz")
print("   • MobileNetV2: Yaprak dokusuna odaklanıyor")
print("   • ResNet50: Bağlam bilgisini (arka plan) kullanıyor")
print("   • EfficientNetB0: Kontrast tabanlı negatif tanı yapıyor")

In [ ]:
# ============================================================
# 5. GÖRSELLERİ DRIVE'A YEDEKLE
# ============================================================

# Üretilen görselleri Drive'a kopyala
gradcam_files = ['gradcam_comparison.png', 'gradcam_healthy.png']

for f in gradcam_files:
    if os.path.exists(f):
        dst = f"{drive_proje}/{f}"
        shutil.copy(f, dst)
        size_kb = os.path.getsize(dst) / 1024
        print(f"✅ {f} Drive'a yedeklendi ({size_kb:.1f} KB)")

print("\n📌 Grad-CAM analizleri Drive'da güvende.")

## ✅ Grad-CAM Analizi Tamamlandı

### 🎯 Akademik Bulgular

#### 1. Hastalıklı Yapraklarda
- **ResNet50**: Hastalık belirtilerine **net odaklanma** gösteriyor
- **EfficientNetB0**: Hastalık bölgelerini doğru tespit ediyor
- **MobileNetV2**: Bazen dağınık odaklanma gösteriyor (zorlandığı sınıflarda)

#### 2. Sağlıklı Yapraklarda — İlginç Bir Bulgu
Modeller farklı stratejiler izliyor:

- **MobileNetV2** → Doğrudan yaprak dokusuna bakıp doku analizi yapıyor
- **ResNet50 ve EfficientNetB0** → Yaprağın "normal" olduğunu **bağlam bilgisinden** çıkarıyor
  (kenarlar, arka plan, yaprak şekli)

Bu, derin modellerin **kontrast tabanlı negatif tanı** yaptığını gösteriyor:
> "Anormal bir şey yok, demek ki sağlıklı."

### 🎓 Akademik Değer

Bu analiz, projemize **açıklanabilir yapay zeka (XAI)** boyutu kazandırıyor:

✅ Model kararları **doğrulanabilir**  
✅ Yanlış tahminlerin sebebi **anlaşılabilir**  
✅ Tarımsal uygulama için **güven** sağlanıyor  
✅ Ziraat uzmanı için **karar destek** anlamı taşıyor  

### 📊 Çıktılar
- `gradcam_comparison.png` — 3 model hastalıklı yaprak karşılaştırması
- `gradcam_healthy.png` — Sağlıklı yaprak analizi

### 🏁 Proje Tamamlandı

Bu notebook **son adımdı**. Artık projemiz tamamlanmış durumda:

✅ Veri hazırlık ve keşif  
✅ 3 modelin eğitimi (MobileNetV2, ResNet50, EfficientNetB0)  
✅ Detaylı karşılaştırma raporu  
✅ Açıklanabilir AI analizi (Grad-CAM)  

### Sıradaki Adım (Proje Dışı)
👉 Streamlit web arayüzü geliştirme (VS Code üzerinde)